## Downloading flood conditioning variables

## Elevation, slope

In [8]:
import ee

import numpy as np
import time

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

def tile_name_from_latlon(lat, lon):

    """
    Example usage:
    Nairobi coordinates: 0.0236° S, 37.9062° E
    print(f"Nairobi tile: {tile_name_from_latlon(lat = 1.29, lon = 36.82)}")
    """

    # Data is in 3 x 3 degree tiles so
     
    lat_tile = (lat // 3) * 3
    lon_tile = (lon // 3) * 3
    lat_tile, lon_tile = int(lat_tile), int(lon_tile)

    lat_prefix = 'N' if lat_tile >= 0 else 'S'
    lon_prefix = 'E' if lon_tile >= 0 else 'W' 

    return f"{lat_prefix}{abs(lat_tile):02d}{lon_prefix}{abs(lon_tile):03d}"

def get_country_bounds(country_name):
    """
    Returns the bounding box of a country as [min_lon, min_lat, max_lon, max_lat]
    """
    countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    bounds = np.array(country.geometry().bounds().getInfo()['coordinates'][0])

    # Get min and max latitudes and longitudes
    min_lon, max_lon = bounds[:, 0].min(), bounds[:, 0].max()
    min_lat, max_lat = bounds[:, 1].min(), bounds[:, 1].max()
    return [min_lon, min_lat, max_lon, max_lat]

def generate_tiles_from_bounds(bounds, tile_size=3):
    """
    Generates a list of tile names that cover the given bounding box.

    Returns:
    A list of tuples, where each tuple contains the bounding box of a tile in the format (min_lon, min_lat, max_lon, max_lat).
    """
    min_lon, min_lat, max_lon, max_lat = bounds

    # Align the bounds to the tile grid
    start_lon = (min_lon // tile_size) * tile_size
    end_lon = (max_lon // tile_size) * tile_size
    start_lat = (min_lat // tile_size) * tile_size
    end_lat = (max_lat // tile_size) * tile_size

    tiles = []
    for lat in range(int(start_lat), int(end_lat) + tile_size, tile_size):
        for lon in range(int(start_lon), int(end_lon) + tile_size, tile_size):
            tile = (lon, lat, lon + tile_size, lat + tile_size)
            tiles.append(tile)
            
    return tiles

def download_data_for_tile(tile, image, dataset_name, export_params):
    """
    Exports data for a given tile and dataset to Google Drive.
    """
    min_lon, min_lat, max_lon, max_lat = tile
    tile_name = tile_name_from_latlon(min_lat, min_lon)
    

    tile_geom = ee.Geometry.Rectangle(tile)
    export_params['region'] = tile_geom

    # Check if a task with the same description already exists
    tasks = ee.batch.Task.list()

    task_exists = any(
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state in ['READY', 'RUNNING'])) or
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state == 'COMPLETED'))
    for t in tasks
    )

    if task_exists:
        print(f"Task for {tile_name} and dataset {dataset_name} already exists. Skipping export.")
        return True

    print(f"Exporting tile: {tile_name} for dataset: {dataset_name}")
    task = ee.batch.Export.image.toDrive(
        image = image,
        description = f"{tile_name}_{dataset_name}",
        **export_params
    )
    task.start()
    return True

def check_task_status(n = 50):
    tasks = ee.batch.Task.list()
    print(f"{'TASK DESCRIPTION':<30} | {'STATE':<10} | {'ID'}")
    print("-" * 60)
    for task in tasks[:n]:  # Check the first n tasks
        status = task.status()
        description = status.get('description', 'No Description')
        state = status.get('state', 'UNKNOWN')
        task_id = status.get('id')
        print(f"{description:<30} | {state:<10} | {task_id}")
    


# Define export parameters
export_params = {
    # 'scale': 30, # Force 30m resolution for ALL layers
    # 'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3' # Creates this folder in Drive
}

In [16]:
# Get the bounding box for Kenya
kenya_bounds = get_country_bounds("Kenya")

# Generate 3x3 degree tiles that cover Kenya
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size=3)
kenya_tiles

[(33, -6, 36, -3),
 (36, -6, 39, -3),
 (39, -6, 42, -3),
 (33, -3, 36, 0),
 (36, -3, 39, 0),
 (39, -3, 42, 0),
 (33, 0, 36, 3),
 (36, 0, 39, 3),
 (39, 0, 42, 3),
 (33, 3, 36, 6),
 (36, 3, 39, 6),
 (39, 3, 42, 6)]

In [ ]:
# Get the bounding box for Kenya
kenya_bounds = get_country_bonds("Kenya")

# Generate 3x3 degree tiles that cover Kenya
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size=3)

# # Define global datasets
# # DEM: SRTM V3 (30m)
# srtm = ee.Image("USGS/SRTMGL1_003")
# dem_global = srtm.select('elevation')

# # Slope: Derived from SRTM
# slope_global = ee.Terrain.slope(dem_global)


def download_elevation_and_slope(tile, export_params, kwargs = None):
    """
    Exports elevation and slope data for a given tile to Google Drive.
    """
    export_params = dict(export_params)  # Ensure it's a dictionary
    if kwargs is not None:
        export_params.update(kwargs)  # Update with any additional parameters

    # Define global datasets
    # DEM: SRTM V3 (30m)
    srtm = ee.Image("USGS/SRTMGL1_003")
    dem_global = srtm.select('elevation')

    # Slope: Derived from SRTM
    slope_global = ee.Terrain.slope(dem_global)

    # Export DEM
    download_data_for_tile(tile, dem_global, "DEM", export_params)
    # Export Slope
    download_data_for_tile(tile, slope_global, "Slope", export_params)
    



# Download DEM and Slope for each tile
for tile in kenya_tiles:
    download_elevation_and_slope(tile, export_params, kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.0002777777777777778, 0, -180.0001388888889, 0, -0.0002777777777777778, 60.00013888888889]})
    time.sleep(10)
    break

# Check task status
check_task_status()


Exporting tile: S06E033 for dataset: DEM
Exporting tile: S06E033 for dataset: Slope
TASK DESCRIPTION               | STATE      | ID
------------------------------------------------------------
S06E033_Slope                  | READY      | FNUSME47G37ZJ3WOHXWVZIAF
S06E033_DEM                    | RUNNING    | XVDVHQPTDM3CUNQ54Y5LFCP2
S06E033_ERA5_SM_sm_14d_mean_2020_04_15 | COMPLETED  | 6XCDJU2DZBGJKJ4IQ2OWJJZL
S06E033_ERA5_SM_daily_mean_2020_04_15 | COMPLETED  | OJJTRCDNFJJ64AYNETFFHWJ3
S06E033_CHIRPS_precip_14d_sum_2020_04_15 | COMPLETED  | 2NTCPEZLHKCJ5WMRUTMGZTDA
S06E033_CHIRPS_precip_2020_04_15 | COMPLETED  | ISBU4IGWCFFL3CXJXS2URF5C


In [63]:

# Define global datasets
# DEM: SRTM V3 (30m)
srtm = ee.Image("USGS/SRTMGL1_003")
dem_global = srtm.select('elevation')

# Slope: Derived from SRTM
slope_global = ee.Terrain.slope(dem_global)


In [67]:
0.0002777777777777778*111320

30.92222222222222

In [65]:
srtm.getInfo()

{'type': 'Image',
 'bands': [{'id': 'elevation',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': -32768,
    'max': 32767},
   'dimensions': [1296001, 417601],
   'crs': 'EPSG:4326',
   'crs_transform': [0.0002777777777777778,
    0,
    -180.0001388888889,
    0,
    -0.0002777777777777778,
    60.00013888888889]}],
 'version': 1641990767055141,
 'id': 'USGS/SRTMGL1_003',
 'properties': {'system:visualization_0_min': '0.0',
  'type_name': 'Image',
  'keywords': ['dem',
   'elevation',
   'geophysical',
   'nasa',
   'srtm',
   'topography',
   'usgs'],
  'thumb': 'https://mw1.google.com/ges/dd/images/SRTM90_V4_thumb.png',
  'description': '<p>The Shuttle Radar Topography Mission (SRTM, see <a href="https://onlinelibrary.wiley.com/doi/10.1029/2005RG000183/full">Farr\net al. 2007</a>)\ndigital elevation data is an international research effort that\nobtained digital elevation models on a near-global scale. This\nSRTM V3 product (SRTM Plus) is provided by NASA JP

## CHIRPS precipitation (daily + rolling aggregation)

Goal: for each Sentinel-1 acquisition date $t$, export CHIRPS precipitation for that day and/or compute revisit-aware rolling aggregates ending at $t$ (e.g., 1/3/7/14-day sums).

In [9]:
from datetime import date as date_type, datetime, timedelta

CHIRPS_DAILY = "UCSB-CHG/CHIRPS/DAILY"

def _to_ee_date(d):
    if isinstance(d, ee.Date):
        return d
    
    if isinstance(d, str):
        # Expect YYYY-MM-DD
        return ee.Date(d)
    
    if isinstance(d, datetime):
        return ee.Date(d.strftime("%Y-%m-%d"))
    
    if isinstance(d, date_type):
        return ee.Date(d.strftime("%Y-%m-%d"))
    
    raise TypeError(f"Unsupported date type: {type(d)}")

def _day_to_str(day):
    if isinstance(day, str):
        return day
    if isinstance(day, (datetime, date_type)):
        return day.strftime("%Y-%m-%d")
    # Fallback: client-side fetch. Prefer passing str/datetime to avoid this.
    return _to_ee_date(day).format("YYYY-MM-dd").getInfo()

def chirps_daily_precipitation(day):
    """Return CHIRPS precipitation image for a single UTC day.

    Args:
        day: 'YYYY-MM-DD', datetime/date, or ee.Date.

    Returns:
        ee.Image with band 'precipitation' (mm/day).
    """
    start = _to_ee_date(day)
    end = start.advance(1, 'day')

    img = (
        ee.ImageCollection(CHIRPS_DAILY)
        .filterDate(start, end)
        .select('precipitation')
        .first()
    )

    return ee.Image(img).rename('precipitation').set({"date": start.format("YYYY-MM-dd")})

def chirps_aggregate(end_day, window_days, reducer = "sum"):
    """Aggregate CHIRPS precipitation over multiple days ending at end_day (inclusive).

     Args:
        end_day: 'YYYY-MM-DD', datetime/date, or ee.Date (window ends here, inclusive)
        window_days: int, e.g. 1, 3, 7, 14
        reducer: 'sum' | 'mean' | 'max'

    Returns:
        ee.Image with one band named like 'precip_14d_sum'. Units:
          - sum: mm over window
          - mean: mm/day
          - max: mm/day (max daily value within window)
    """
    if window_days < 1:
        raise ValueError("window_days must be >= 1")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-window_days, 'day')

    ic = (
        ee.ImageCollection(CHIRPS_DAILY)
        .filterDate(start, end)
        .select('precipitation')
    )

    reducer = reducer.lower() # Make reducer lowercase for easier handling

    if reducer == "sum":
        agg = ic.sum()

    elif reducer == "mean":
        agg = ic.mean()

    elif reducer == "max":
        agg = ic.max()

    else:
        raise ValueError(f"Unsupported reducer: {reducer}. Use 'sum', 'mean', or 'max'.")

    band_name = f"precip_{int(window_days)}d_{reducer}"

    return ee.Image(agg).rename(band_name).set({
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end_og.format("YYYY-MM-dd"),
        "window_days": window_days,
        "reducer": reducer
    })


def download_chirps_precipitation_for_day(tile, day, export_params, dataset_prefix = "CHIRPS_precip", kwargs = None):
    """Export CHIRPS precipitation for a single day for the given 3°×3° tile.

    Notes:
      - We override scale to CHIRPS native (~0.05° ≈ 5.5km). Do NOT export at 30m.
      - Uses the existing download_data_for_tile() helper.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        day: 'YYYY-MM-DD' or datetime/date
        export_params: dict used by ee.batch.Export.image.toDrive
        scale_m: export scale in meters (default ~5.5km)
        dataset_prefix: name prefix used in Drive description
    """
    # assert scale_m is not None, "Must specify scale_m for CHIRPS export (e.g. 5500 for ~5.5km native resolution)"
    day_str = _day_to_str(day)
    image = chirps_daily_precipitation(day)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)
   

    dataset_name = f"{dataset_prefix}_{day_str.replace('-', '_')}"

    return download_data_for_tile(tile, image, dataset_name, export_params)

    
def download_chirps_precipitation_aggregate(tile, end_day, window_days, reducer, export_params, dataset_prefix="CHIRPS", kwargs=None):
    """Export a CHIRPS rolling-window aggregate ending at end_day for the given tile.

    Example dataset_name: CHIRPS_precip_14d_sum_2020_04_15

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        end_day: 'YYYY-MM-DD' or datetime/date
        window_days: e.g. 14
        reducer: 'sum' | 'mean' | 'max'
        export_params: dict used by ee.batch.Export.image.toDrive
        scale_m: export scale in meters (default ~5.5km)
    """
    
    end_str = _day_to_str(end_day)
    image = chirps_aggregate(end_day, window_days, reducer = reducer)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_precip_{window_days}d_{reducer}_{end_str.replace('-', '_')}"

    return download_data_for_tile(tile, image, dataset_name, export_params)



# Example usage (uncomment):
# What's best reducer?
# tile = kenya_tiles[0]
# download_chirps_precipitation_for_day(tile, "2020-04-15", export_params, kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.05, 0, -180, 0, -0.05, 50]})
# download_chirps_precipitation_aggregate(tile, "2020-04-15", window_days=14, reducer="sum", export_params=export_params, kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.05, 0, -180, 0, -0.05, 50]})


## ERA5-Land soil moisture (hourly → daily snapshot + rolling aggregates)

Goal: for each Sentinel-1 acquisition date $t$, export soil moisture features aligned to the pass date.

- Default: **daily mean** of hourly ERA5-Land on date $t$ (robust when acquisition time is unknown)
- Optional: rolling-window means (e.g., 14-day mean ending at $t$) and lag-deltas (e.g., $SM(t)-SM(t-7)$)

In [61]:
ERA5_LAND_HOURLY = "ECMWF/ERA5_LAND/HOURLY"

# Volumetric soil water at 4 depth layers
ERA5_SM_BANDS = [
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "volumetric_soil_water_layer_4",
]

def era5_sm_daily_mean_image(day):
    """Daily mean soil moisture for the given UTC day.

    Uses hourly ERA5-Land, averages all hours within [day, day+1).

    Returns an ee.Image with bands: sm_l1, sm_l2, sm_l3, sm_l4
    """
    start = _to_ee_date(day)
    end = start.advance(1, 'day')

    img = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select(ERA5_SM_BANDS)
        .mean()
    )

    return ee.Image(img).rename(["sm_l1", "sm_l2", "sm_l3", "sm_l4"]).set({
        "date": start.format("YYYY-MM-dd"),
        "aggregation": "daily_mean",
    })

def era5_sm_aggregate(end_day, window_days, reducer = "mean"):
    """Aggregate ERA5-Land hourly soil moisture over a rolling window ending at end_day (inclusive).

    Note: For soil moisture, 'mean' over the window is usually the most sensible.

    Args:
        end_day: window ends here (inclusive)
        window_days: number of days in window (>=1)
        reducer: 'mean' | 'max' | 'min'

    Returns:
        ee.Image with 4 bands named like: sm_l1_14d_mean, ...
    """
    if window_days < 1:
        raise ValueError("window_days must be >= 1")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-window_days, 'day')

    ic = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select(ERA5_SM_BANDS)

    )
    reducer = reducer.lower()
    if reducer == "sum":
        agg = ic.sum()
    elif reducer == "mean":
        agg = ic.mean()
    elif reducer == "max":
        agg = ic.max()
    else:
        raise ValueError("reducer must be one of: 'sum', 'mean', 'max'")
    
    band_names = [f"sm_l{i+1}_{window_days}d_{reducer}" for i in range(4)]
    return ee.Image(agg).rename(band_names).set({
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end_og.format("YYYY-MM-dd"),
        "window_days": window_days,
        "reducer": reducer
    })

def download_era5_sm_for_day(tile, day, export_params, dataset_prefix = "ERA5_SM", kwargs = None):
    """Export ERA5-Land soil moisture for a single day for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        day: 'YYYY-MM-DD' or datetime/date
        export_params: dict used by ee.batch.Export.image.toDrive
        dataset_prefix: name prefix used in Drive description
    """
    day_str = _day_to_str(day)
    image = era5_sm_daily_mean_image(day)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_daily_mean_{day_str.replace('-', '_')}"

    return download_data_for_tile(tile, image, dataset_name, export_params)

def download_era5_sm_aggregate(tile, end_day, window_days, reducer, export_params, dataset_prefix="ERA5_SM", kwargs=None):
    """Export ERA5-Land soil moisture aggregate for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        end_day: window ends here (inclusive)
        window_days: number of days in window (>=1)
        reducer: 'mean' | 'max' | 'min'
        export_params: dict used by ee.batch.Export.image.toDrive
        dataset_prefix: name prefix used in Drive description
    """
    
    end_str = _day_to_str(end_day)
    image = era5_sm_aggregate(end_day, window_days, reducer = reducer)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{window_days}d_{reducer}_{end_str.replace('-', '_')}"

    return download_data_for_tile(tile, image, dataset_name, export_params)

# # Example usage (uncomment):
# tile = kenya_tiles[0]
# download_era5_sm_for_day(tile, "2020-04-15", export_params, dataset_prefix="ERA5_SM", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})
# download_era5_sm_aggregate(tile, "2020-04-15", window_days=14, reducer="mean", export_params=export_params, dataset_prefix="ERA5_SM", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})


Exporting tile: S06E033 for dataset: ERA5_SM_daily_mean_2020_04_15
Exporting tile: S06E033 for dataset: ERA5_SM_sm_14d_mean_2020_04_15


True

## ERA5-Land temperature (hourly → daily mean + rolling aggregates)

Goal: export pass-date aligned temperature features for each Sentinel-1 acquisition date $t$.

- Default: **daily mean** of hourly ERA5-Land on date $t$
- Optional: rolling-window means/min/max ending at $t$ for short-term thermal context

In [ ]:
ERA5_LAND_HOURLY = "ECMWF/ERA5_LAND/HOURLY"

def era5_temperature_daily_mean_image(day):
    """Daily mean 2m temperature for the given UTC day."""
    start = _to_ee_date(day)
    end = start.advance(1, 'day')

    img = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select('temperature_2m')
        .mean()
    )

    return ee.Image(img).rename(['temperature_2m']).set({
        "date": start.format("YYYY-MM-dd"),
        "aggregation": "daily_mean",
    })


def era5_temperature_aggregate(end_day, window_days, reducer = "mean"):
    """Aggregate 2m temperature over a rolling window ending at end_day (inclusive)."""
    if window_days < 1:
        raise ValueError("window_days must be >= 1")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-window_days, 'day')

    ic = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select('temperature_2m')
    )

    reducer = reducer.lower()
    if reducer == "mean":
        agg = ic.mean()
    elif reducer == "min":
        agg = ic.min()
    elif reducer == "max":
        agg = ic.max()
    else:
        raise ValueError("reducer must be one of: 'mean', 'min', 'max'")
    
    band_name = f"temperature_2m_{window_days}d_{reducer}"
    return ee.Image(agg).rename(band_name).set({
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end_og.format("YYYY-MM-dd"),
        "window_days": window_days,
        "reducer": reducer
    })

def download_era5_temperature_for_day(tile, day, export_params, dataset_prefix = "ERA5_Temp", kwargs = None):
    """Export ERA5-Land 2m temperature for a single day for the given tile."""
    day_str = _day_to_str(day)
    image = era5_temperature_daily_mean_image(day)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_daily_mean_{day_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)

def download_era5_temperature_aggregate(tile, end_day, window_days, reducer, export_params, dataset_prefix="ERA5_Temp", kwargs=None):
    """Export ERA5-Land 2m temperature aggregate for the given tile."""
    end_str = _day_to_str(end_day)
    image = era5_temperature_aggregate(end_day, window_days, reducer = reducer)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{window_days}d_{reducer}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)

# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_era5_temperature_for_day(tile, "2020-04-15", export_params, dataset_prefix="ERA5_Temp", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})
# download_era5_temperature_aggregate(tile, "2020-04-15", window_days=14, reducer="mean", export_params=export_params, dataset_prefix="ERA5_Temp", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})

## ERA5-Land runoff (hourly → daily sums + rolling aggregates)

Goal: export pass-date aligned hydrologic fluxes for each Sentinel-1 acquisition date $t$.

- Default: **daily sums** for `surface_runoff`, `runoff`, `total_precipitation`, and `total_evaporation`
- Optional: rolling-window sums/means/max over 3/7/14 days ending at $t$

In [ ]:
ERA5_LAND_HOURLY = "ECMWF/ERA5_LAND/HOURLY"
ERA5_RUNOFF_BANDS = [
    "surface_runoff",
    "runoff",

]

def era5_runoff_daily_sum_image(day):
    """Daily sum runoff for the given UTC day."""
    start = _to_ee_date(day)
    end = start.advance(1, 'day')

    img = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select(ERA5_RUNOFF_BANDS)
        .sum()
    )

    return ee.Image(img).rename(["surface_runoff", "runoff"]).set({
        "date": start.format("YYYY-MM-dd"),
        "aggregation": "daily_sum",
    })

def era5_runoff_aggregate(end_day, window_days, reducer = "sum"):
    """Aggregate runoff over a rolling window ending at end_day (inclusive)."""
    if window_days < 1:
        raise ValueError("window_days must be >= 1")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-window_days, 'day')

    ic = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select(ERA5_RUNOFF_BANDS)
    )

    reducer = reducer.lower()
    if reducer == "sum":
        agg = ic.sum()
    elif reducer == "mean":
        agg = ic.mean()
    elif reducer == "max":
        agg = ic.max()
    else:
        raise ValueError("reducer must be one of: 'sum', 'mean', 'max'")
    
    band_names = [f"{band}_{window_days}d_{reducer}" for band in ERA5_RUNOFF_BANDS]
    return ee.Image(agg).rename(band_names).set({
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end_og.format("YYYY-MM-dd"),
        "window_days": window_days,
        "reducer": reducer
    })

def download_era5_runoff_for_day(tile, day, export_params, dataset_prefix = "ERA5_Runoff", kwargs = None):
    """Export ERA5-Land runoff for a single day for the given tile."""
    day_str = _day_to_str(day)
    image = era5_runoff_daily_sum_image(day)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_daily_sum_{day_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)


def download_era5_runoff_aggregate(tile, end_day, window_days, reducer, export_params, dataset_prefix="ERA5_Runoff", kwargs=None):
    """Export ERA5-Land runoff aggregate for the given tile."""
    end_str = _day_to_str(end_day)
    image = era5_runoff_aggregate(end_day, window_days, reducer = reducer)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{window_days}d_{reducer}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)

# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_era5_runoff_for_day(tile, "2020-04-15", export_params, dataset_prefix="ERA5_Runoff", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})
# download_era5_runoff_aggregate(tile, "2020-04-15", window_days=14, reducer="sum", export_params=export_params, dataset_prefix="ERA5_Runoff", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.1, 0, -180.05, 0, -0.1, 90.05]})

## MODIS NDVI (16-day composites aligned to Sentinel-1 dates)

Goal: export vegetation context for each Sentinel-1 acquisition date $t$.

Because MODIS NDVI is a **16-day composite** rather than daily, we use a **causal recent-composite strategy**:

- default: most recent available MODIS NDVI composite on or before date $t$
- optional: aggregate all MODIS NDVI composites within a lookback window ending at $t$

In [10]:
#MODIS_NDVI_COLLECTION = "MODIS/061/MOD13A2"
MODIS_NDVI_COLLECTION = "MODIS/061/MOD13Q1"
MODIS_NDVI_SCALE = 0.0001


def modis_ndvi_recent_image(end_day, lookback_days = 32):
    """Get the most recent MODIS NDVI image within [end_day - lookback_days, end_day].

    Note: MODIS NDVI is every 16 days, so we look back ~32 days to have a good chance of getting at least one image.

    Returns:
        ee.Image with band 'NDVI' scaled to [0, 1].
    """
    if lookback_days < 16:
        raise ValueError("lookback_days should usually be >= 16 for MODIS NDVI")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-lookback_days, 'day')

    img = (
        ee.ImageCollection(MODIS_NDVI_COLLECTION)
        .filterDate(start, end)
        .select('NDVI')
        .sort('system:time_start', False) # Sort by time descending to get the most recent image first
        .first()
    )

    return ee.Image(img).multiply(MODIS_NDVI_SCALE).rename(['ndvi']).set({
        'end_date': end_og.format('YYYY-MM-dd'),
        'lookback_days': lookback_days,
        'selection': 'most_recent_composite',
    })

def modis_ndvi_aggregate(end_day, lookback_days = 32, reducer = "mean"):
    """Aggregate MODIS NDVI over images within [end_day - lookback_days, end_day].

    Args:
        end_day: window ends here (inclusive)
        lookback_days: number of days to look back for MODIS images
        reducer: 'mean' | 'max' | 'min'

    Returns:
        ee.Image with band named like 'ndvi_32d_mean'.
    """
    if lookback_days < 16:
        raise ValueError("lookback_days should usually be >= 16 for MODIS NDVI")
    
    end_og = _to_ee_date(end_day)
    end = _to_ee_date(end_day).advance(1, 'day')  # Advance by 1 day to make end_day inclusive
    start = end.advance(-lookback_days, 'day')

    ic = (
        ee.ImageCollection(MODIS_NDVI_COLLECTION)
        .filterDate(start, end)
        .select('NDVI')
        .map(lambda img: ee.Image(img).multiply(MODIS_NDVI_SCALE))

    )

    reducer = reducer.lower()
    if reducer == "mean":
        agg = ic.mean()
    elif reducer == "max":
        agg = ic.max()
    elif reducer == "min":
        agg = ic.min()
    else:
        raise ValueError("reducer must be one of: 'mean', 'max', 'min'")
    
    band_name = f"ndvi_{lookback_days}d_{reducer}"
    return ee.Image(agg).rename(band_name).set({
        'end_date': end_og.format('YYYY-MM-dd'),
        'lookback_days': lookback_days,
        'reducer': reducer,
    })


def download_modis_ndvi_for_day(tile, end_day, export_params, lookback_days = 32, dataset_prefix = "MODIS_NDVI", kwargs = None):
    """Export the most recent MODIS NDVI image for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        end_day: 'YYYY-MM-DD' or datetime/date (window ends here, inclusive)
        export_params: dict used by ee.batch.Export.image.toDrive
        lookback_days: number of days to look back for MODIS images
        dataset_prefix: name prefix used in Drive description
    """
    end_str = _day_to_str(end_day)
    image = modis_ndvi_recent_image(end_day, lookback_days)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_recent_{lookback_days}d_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)


def download_modis_ndvi_aggregate(tile, end_day, lookback_days, reducer, export_params, dataset_prefix="MODIS_NDVI", kwargs=None):
    """Export aggregated MODIS NDVI for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        end_day: 'YYYY-MM-DD' or datetime/date (window ends here, inclusive)
        lookback_days: number of days to look back for MODIS images
        reducer: 'mean' | 'max' | 'min'
        export_params: dict used by ee.batch.Export.image.toDrive
        dataset_prefix: name prefix used in Drive description
    """
    end_str = _day_to_str(end_day)
    image = modis_ndvi_aggregate(end_day, lookback_days, reducer)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{lookback_days}d_{reducer}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params)
    
# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_modis_ndvi_for_day(tile, "2020-04-15", export_params, lookback_days=32, dataset_prefix="MODIS_NDVI", kwargs = {"crs": 'EPSG:4326', "scale": 926.625433})
# download_modis_ndvi_aggregate(tile, "2020-04-15", lookback_days=32, reducer="mean", export_params=export_params, dataset_prefix="MODIS_NDVI", kwargs = {"crs": 'EPSG:4326', "scale": 926.625433})
 
    
        

In [9]:
export_params

{'maxPixels': 10000000000000.0,
 'fileFormat': 'GeoTIFF',
 'folder': 'Kenya_Flood_Data_3x3'}

In [6]:
modis_ndvi_recent_image("2020-04-15").getInfo()

{'type': 'Image',
 'bands': [{'id': 'ndvi',
   'data_type': {'type': 'PixelType',
    'precision': 'double',
    'min': -3.2768,
    'max': 3.2767},
   'dimensions': [43200, 18000],
   'crs': 'SR-ORG:6974',
   'crs_transform': [926.625433055833,
    0,
    -20015109.354,
    0,
    -926.6254330558334,
    10007554.677003]}],
 'properties': {'end_date': '2020-04-15',
  'selection': 'most_recent_composite',
  'lookback_days': 32}}

In [11]:
modis_ndvi_recent_image("2020-04-15").getInfo()

{'type': 'Image',
 'bands': [{'id': 'ndvi',
   'data_type': {'type': 'PixelType',
    'precision': 'double',
    'min': -3.2768,
    'max': 3.2767},
   'dimensions': [172800, 72000],
   'crs': 'SR-ORG:6974',
   'crs_transform': [231.65635826395825,
    0,
    -20015109.354,
    0,
    -231.65635826395834,
    10007554.677003]}],
 'properties': {'end_date': '2020-04-15',
  'selection': 'most_recent_composite',
  'lookback_days': 32}}

In [ ]:
import ee

# 1. Define your area of interest (min_lon, min_lat, max_lon, max_lat)
# Example: Using the tile you mentioned earlier
roi = ee.Geometry.Rectangle([33, -6, 36, -3])

# 2. Set up the export task
task = ee.batch.Export.image.toDrive(
    image = ndvi_image,             # Your MODIS NDVI image object
    description = 'MODIS_NDVI_Export',
    folder = 'GEE_Flood_Data',
    fileNamePrefix = 'ndvi_2020_04_15',
    region = roi,                   # The boundary of the download
    scale = 926.625433,             # The native resolution from your metadata
    crs = 'EPSG:4326',              # Convert to standard Lat/Lon
    maxPixels = 1e13                # High limit to avoid "too many pixels" error
)

# 3. Start the task
task.start()

print("Export started. Check your Google Drive 'GEE_Flood_Data' folder.")

## Static soil properties (SoilGrids-style layers via Earth Engine OpenLandMap)

Goal: export static soil conditioning layers once per tile.

Practical choice: use Earth Engine-accessible **OpenLandMap** soil-property rasters as SoilGrids-style static predictors.

Included here:
- **Clay content** at 250 m

In [ ]:
SOIL_STATIC_ASSETS = {
    'clay': {
        'asset': 'OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02',
        'bands': ['b0', 'b10', 'b30', 'b60'],
        'rename_prefix': 'clay',
    },
}

def soil_static_property_image(property_name = 'clay'):
    spec = SOIL_STATIC_ASSETS[property_name]
    bands = spec['bands']
    rename = [f"{spec['rename_prefix']}_{band}" for band in bands]

    img = ee.Image(spec['asset']).select(bands).rename(rename)
    return img.set({
        'property_name': property_name,
        'static_layer': True,

    })

def download_soil_static_property(tile, export_params, property_name = 'clay', dataset_prefix = "Soil_Static", kwargs = None):
    """Export a soil static property for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        export_params: dict used by ee.batch.Export.image.toDrive
        property_name: e.g. 'clay'
        dataset_prefix: name prefix used in Drive description
    """
    image = soil_static_property_image(property_name)

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{property_name}"
    return download_data_for_tile(tile, image, dataset_name, export_params)

# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_soil_static_property(tile, export_params, property_name='clay', dataset_prefix="Soil_Static", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.002083333, 0, -180, 0, -0.002083333, 87.37]})


## Static hydrological conditioning layers

Goal: export static hydrological conditioning layers per tile, derived from **MERIT Hydro** (`MERIT/Hydro/v1_0_1`, 3 arc-second / ~90 m):

1. **Flow accumulation** — upstream drainage area (`upa` band, in km²). Identifies where water concentrates. Log-transformed for better dynamic range.
2. **Distance to rivers** — Euclidean distance (in metres) from each pixel to the nearest "river" pixel, where rivers are defined by thresholding upstream area (≥ 10 km²). High values = far from drainage network = less fluvial flood risk.
3. **Height Above Nearest Drainage (HAND)** — bonus: vertical distance to nearest drainage. Arguably the single best flood susceptibility predictor after elevation.

In [ ]:
MERIT_HYDRO = "MERIT/Hydro/v1_0_1"

# ── Flow accumulation (upstream drainage area) ──────────────────────────────────
def merit_flow_accumulation_image(perform_log_transform = True):
    """Return MERIT Hydro upstream drainage area (km²), log10-transformed.

    Band `upa` gives the upstream drainage area in km² for every ~90 m pixel.
    We apply log10(upa + 1) so the huge dynamic range (0 → millions km²)
    compresses into a model-friendly range (~0–6).

    Returns:
        ee.Image with band 'flow_acc_log10' (unitless, log10 km²).
    """
    merit = ee.Image(MERIT_HYDRO)
    upa = merit.select('upa') # Upstream drainage area in km²
    if perform_log_transform:
        flow_acc_log10 = upa.add(1).log10().rename('flow_acc_log10')
        return flow_acc_log10.set({"source": "MERIT_Hydro", "band": "upa", "transform": "log10(upa+1)", "static_layer": True})
    else:
        return upa.set({"source": "MERIT_Hydro", "band": "upa", "static_layer": True})

# ── Height Above Nearest Drainage (HAND) ────────────────────────────────────────
def merit_hand_image():
    """Return MERIT Hydro Height Above Nearest Drainage (HAND) in metres.

    HAND is the vertical distance between each pixel and the nearest river
    pixel along the drainage path.  It is one of the strongest flood
    susceptibility predictors: low HAND = near river level = flood-prone.

    Returns:
        ee.Image with band 'hand_m' (metres).
    """
    merit = ee.Image(MERIT_HYDRO)
    hand = merit.select('hnd').rename('hand_m') # HAND in metres
    return hand.set({"source": "MERIT_Hydro", "band": "hand", "static_layer": True})

def download_merit_hydro_layer(tile, export_params, layer_name = 'flow_accumulation', dataset_prefix = "MERIT_Hydro", perform_log_transform = True, kwargs = None):
    """Export a MERIT Hydro layer for the given tile.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        export_params: dict used by ee.batch.Export.image.toDrive
        layer_name: 'flow_accumulation' or 'hand'
        dataset_prefix: name prefix used in Drive description
    """
    if layer_name == 'flow_accumulation':
        image = merit_flow_accumulation_image(perform_log_transform = perform_log_transform)
    elif layer_name == 'hand':
        image = merit_hand_image()
    else:
        raise ValueError("layer_name must be 'flow_accumulation' or 'hand'")

    export_params = dict(export_params)
    if kwargs is not None:
        export_params.update(kwargs)

    dataset_name = f"{dataset_prefix}_{layer_name}"
    return download_data_for_tile(tile, image, dataset_name, export_params)

# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_merit_hydro_layer(tile, export_params, layer_name='flow_accumulation', dataset_prefix="MERIT_Hydro", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.0008333333333333334,0,-180.00041666666667,0,-0.0008333333333333334,84.99958333333333]})
# download_merit_hydro_layer(tile, export_params, layer_name='hand', dataset_prefix="MERIT_Hydro", kwargs = {"crs": 'EPSG:4326', "crsTransform": [0.0008333333333333334,0,-180.00041666666667,0,-0.0008333333333333334,84.99958333333333]})

In [ ]:
[0.0008333333333333334,0,-180.00041666666667,0,-0.0008333333333333334,84.99958333333333]

In [11]:
merit_flow_accumulation_image().projection().nominalScale().getInfo()


92.76624232772798

In [7]:
merit_hand_image().getInfo()

{'type': 'Image',
 'bands': [{'id': 'hand_m',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [432000, 174000],
   'crs': 'EPSG:4326',
   'crs_transform': [0.0008333333333333334,
    0,
    -180.00041666666667,
    0,
    -0.0008333333333333334,
    84.99958333333333]}],
 'version': 1648043364539823,
 'id': 'MERIT/Hydro/v1_0_1',
 'properties': {'static_layer': True,
  'band': 'hand',
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-180, -90],
    [180, -90],
    [180, 90],
    [-180, 90],
    [-180, -90]]},
  'source': 'MERIT_Hydro',
  'system:asset_size': 200458990544}}

In [8]:
0.000833333333333333*111320

92.76666666666662

In [81]:
imgg = soil_static_property_image('clay')
imgg.getInfo()

{'type': 'Image',
 'bands': [{'id': 'clay_b0',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 255},
   'dimensions': [172800, 71698],
   'crs': 'EPSG:4326',
   'crs_transform': [0.002083333, 0, -180, 0, -0.002083333, 87.37]},
  {'id': 'clay_b10',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 255},
   'dimensions': [172800, 71698],
   'crs': 'EPSG:4326',
   'crs_transform': [0.002083333, 0, -180, 0, -0.002083333, 87.37]},
  {'id': 'clay_b30',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 255},
   'dimensions': [172800, 71698],
   'crs': 'EPSG:4326',
   'crs_transform': [0.002083333, 0, -180, 0, -0.002083333, 87.37]},
  {'id': 'clay_b60',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 255},
   'dimensions': [172800, 71698],
   'crs': 'EPSG:4326',
   'crs_transform': [0.002083333, 0, -180, 0, -0.002083333, 87.37]}],
 'version': 1

In [73]:
era5_runoff_aggregate("2020-04-15", window_days=14, reducer="sum").getInfo()

{'type': 'Image',
 'bands': [{'id': 'surface_runoff_14d_sum',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'runoff_14d_sum',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}],
 'properties': {'end_date': '2020-04-15',
  'reducer': 'sum',
  'start_date': '2020-04-02',
  'window_days': 14}}

In [70]:
era5_temperature_daily_mean_image("2020-04-15").getInfo()

{'type': 'Image',
 'bands': [{'id': 'temperature_2m',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}],
 'properties': {'date': '2020-04-15', 'aggregation': 'daily_mean'}}

In [57]:
era5_sm_aggregate(end_day="2020-04-15", window_days=14, reducer="mean").getInfo()

{'type': 'Image',
 'bands': [{'id': 'sm_l1_14d_mean',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l2_14d_mean',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l3_14d_mean',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l4_14d_mean',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}],
 'properties': {'end_date': '2020-04-15',
  'reducer': 'mean',
  'start_date': '2020-04-02',
  'window_days': 14}}

In [55]:
img = ee.ImageCollection(ERA5_LAND_HOURLY).filterDate("2020-04-15", "2020-04-16").select(ERA5_SM_BANDS)
img.getInfo()

{'type': 'ImageCollection',
 'bands': [],
 'version': 1775288293013026,
 'id': 'ECMWF/ERA5_LAND/HOURLY',
 'properties': {'type_name': 'ImageCollection',
  'keywords': ['cds',
   'climate',
   'copernicus',
   'ecmwf',
   'era5-land',
   'evaporation',
   'heat',
   'lakes',
   'precipitation',
   'pressure',
   'radiation',
   'reanalysis',
   'runoff',
   'snow',
   'soil_water',
   'temperature',
   'vegetation',
   'wind'],
  'visualization_1_bands': 'total_precipitation',
  'visualization_1_max': '0.1',
  'description': '<p>ERA5-Land is a reanalysis dataset providing a consistent view of the evolution of land variables\nover several decades at an enhanced resolution compared to ERA5. ERA5-Land has been produced by\nreplaying the land component of the ECMWF ERA5 climate reanalysis. Reanalysis combines model\ndata with observations from across the world into a globally complete and consistent dataset\nusing the laws of physics. Reanalysis produces data that goes several decades back 

In [53]:
ee.Image(img).rename(["sm_l1", "sm_l2", "sm_l3", "sm_l4"]).set({
        "date": _to_ee_date("2020-04-15").format("YYYY-MM-dd"),
        "aggregation": "daily_mean",
    }).getInfo()

{'type': 'Image',
 'bands': [{'id': 'sm_l1',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l2',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l3',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'sm_l4',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}],
 'properties': {'date': '2020-04-15', 'aggregation': 'daily_mean'}}

In [51]:
ee.Image(img).getInfo()

{'type': 'Image',
 'bands': [{'id': 'volumetric_soil_water_layer_1',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'volumetric_soil_water_layer_2',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'volumetric_soil_water_layer_3',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]},
  {'id': 'volumetric_soil_water_layer_4',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}]}

In [46]:
dict_a = {'a': 1, 'b': 2, 'c': 3}
dict_b = {'b': 12, 'c': 13}

dict_a.update(dict_b)

print(dict_a)
# Output: {'a': 1, 'b': 3, 'c': 4}

{'a': 1, 'b': 12, 'c': 13}


In [44]:
export_params

{'scale': 30,
 'crs': 'EPSG:4326',
 'maxPixels': 10000000000000.0,
 'fileFormat': 'GeoTIFF',
 'folder': 'Kenya_Flood_Data_3x3'}

In [33]:
# tRY OUT
d = '2020-04-15'
cd = chirps_daily_precipitation(d)
cd.getInfo()

{'type': 'Image',
 'bands': [{'id': 'precipitation',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [7200, 2000],
   'crs': 'EPSG:4326',
   'crs_transform': [0.05, 0, -180, 0, -0.05, 50]}],
 'version': 1747259059838602,
 'id': 'UCSB-CHG/CHIRPS/DAILY/20200415',
 'properties': {'date': '2020-04-15',
  'system:time_start': 1586908800000,
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-180, -90],
    [180, -90],
    [180, 90],
    [-180, 90],
    [-180, -90]]},
  'system:time_end': 1586995200000,
  'system:asset_size': 4825324,
  'system:index': '20200415'}}

In [36]:
ca = chirps_aggregate(end_day = '2020-04-15', window_days = 1, reducer = "sum")
ca.getInfo()

{'type': 'Image',
 'bands': [{'id': 'precip_1d_sum',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'crs': 'EPSG:4326',
   'crs_transform': [1, 0, 0, 0, 1, 0]}],
 'properties': {'end_date': '2020-04-15',
  'reducer': 'sum',
  'start_date': '2020-04-15',
  'window_days': 1}}

In [42]:
# Example usage (uncomment):
tile = kenya_tiles[0]
download_chirps_precipitation_for_day(tile, "2020-04-15", export_params, scale_m=5566)
download_chirps_precipitation_aggregate(tile, "2020-04-15", window_days=14, reducer="sum", scale_m=5566, export_params=export_params)

Exporting tile: S06E033 for dataset: CHIRPS_precip_2020_04_15
Exporting tile: S06E033 for dataset: CHIRPS_precip_14d_sum_2020_04_15


True

In [8]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filter(
    ee.Filter.date('2018-05-01', '2018-05-03')
)
precipitation = dataset.select('precipitation')
precipitation.getInfo()

{'type': 'ImageCollection',
 'bands': [],
 'version': 1773791573533451,
 'id': 'UCSB-CHG/CHIRPS/DAILY',
 'properties': {'system:visualization_0_min': '1.0',
  'type_name': 'ImageCollection',
  'keywords': ['chg',
   'climate',
   'geophysical',
   'precipitation',
   'ucsb',
   'weather'],
  'thumb': 'https://mw1.google.com/ges/dd/images/CHIRPS_thumb.png',
  'description': '<p>Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS)\nis a 30+ year quasi-global rainfall dataset. CHIRPS incorporates\n0.05° resolution satellite imagery with in-situ station data\nto create gridded rainfall time series for trend analysis and seasonal\ndrought monitoring.</p><p><b>Provider: <a href="https://chc.ucsb.edu/data/chirps">UCSB/CHG</a></b><br><p><b>Resolution</b><br>5566 meters\n</p><p><b>Cadence</b><br>\n  1 day\n</p><p><b>Bands</b><table class="eecat"><tr><th scope="col">Name</th><th scope="col">Description</th></tr><tr><td>precipitation</td><td><p>Precipitation</p></td></tr></tabl

In [39]:
export_params = {
    'scale': 30, # Force 30m resolution for ALL layers
    'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3' # Creates this folder in Drive
}

In [23]:
precipitation.projection()

AttributeError: 'ImageCollection' object has no attribute 'projection'

In [12]:
img = precipitation.first().getInfo()

In [22]:
# Obtain image scale from metadata in meters per pixel
img.projection().getInfo()

AttributeError: 'dict' object has no attribute 'projection'

In [ ]:
from datetime import date as date_type, datetime, timedelta

CHIRPS_DAILY = "UCSB-CHG/CHIRPS/DAILY"


def _to_ee_date(d):
    if isinstance(d, ee.Date):
        return d
    if isinstance(d, str):
        # Expect YYYY-MM-DD
        return ee.Date(d)
    if isinstance(d, datetime):
        return ee.Date(d.strftime("%Y-%m-%d"))
    if isinstance(d, date_type):
        return ee.Date(d.strftime("%Y-%m-%d"))
    raise TypeError(f"Unsupported date type: {type(d)}")


def _day_to_str(day):
    if isinstance(day, str):
        return day
    if isinstance(day, (datetime, date_type)):
        return day.strftime("%Y-%m-%d")
    # Fallback: client-side fetch. Prefer passing str/datetime to avoid this.
    return _to_ee_date(day).format("YYYY-MM-dd").getInfo()


def chirps_daily_image(day):
    """Return CHIRPS precipitation image for a single UTC day.

    Args:
        day: 'YYYY-MM-DD', datetime/date, or ee.Date.

    Returns:
        ee.Image with band 'precipitation' (mm/day).
    """
    start = _to_ee_date(day)
    end = start.advance(1, "day")

    img = (
        ee.ImageCollection(CHIRPS_DAILY)
        .filterDate(start, end)
        .select("precipitation")
        .first()
    )

    # If the collection is empty for a day (rare), this will error later; that's OK.
    return ee.Image(img).rename("precipitation").set({"date": start.format("YYYY-MM-dd")})


def chirps_aggregate(end_day, window_days, reducer="sum"):
    """Aggregate CHIRPS precipitation over a rolling window ending at end_day (inclusive).

    This is the recommended representation for Sentinel-1 pass-date targets.

    Args:
        end_day: 'YYYY-MM-DD', datetime/date, or ee.Date (window ends here, inclusive)
        window_days: int, e.g. 1, 3, 7, 14
        reducer: 'sum' | 'mean' | 'max'

    Returns:
        ee.Image with one band named like 'precip_14d_sum'. Units:
          - sum: mm over window
          - mean: mm/day
          - max: mm/day (max daily value within window)
    """
    if window_days < 1:
        raise ValueError("window_days must be >= 1")

    end = _to_ee_date(end_day)
    start = end.advance(-(window_days - 1), "day")

    ic = (
        ee.ImageCollection(CHIRPS_DAILY)
        .filterDate(start, end.advance(1, "day"))
        .select("precipitation")
    )

    reducer = reducer.lower()
    if reducer == "sum":
        agg = ic.sum()
    elif reducer == "mean":
        agg = ic.mean()
    elif reducer == "max":
        agg = ic.max()
    else:
        raise ValueError("reducer must be one of: 'sum', 'mean', 'max'")

    band_name = f"precip_{int(window_days)}d_{reducer}"
    return ee.Image(agg).rename(band_name).set({
        "end_date": end.format("YYYY-MM-dd"),
        "start_date": start.format("YYYY-MM-dd"),
        "window_days": int(window_days),
        "reducer": reducer,
    })


def download_chirps_for_day(tile, day, export_params, scale_m=5566, dataset_prefix="CHIRPS"):
    """Export CHIRPS precipitation for a single day for the given 3°×3° tile.

    Notes:
      - We override scale to CHIRPS native (~0.05° ≈ 5.5km). Do NOT export at 30m.
      - Uses the existing download_data_for_tile() helper.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        day: 'YYYY-MM-DD' or datetime/date
        export_params: dict used by ee.batch.Export.image.toDrive
        scale_m: export scale in meters (default ~5.5km)
        dataset_prefix: name prefix used in Drive description
    """
    day_str = _day_to_str(day)
    image = chirps_daily_image(day)

    export_params_local = dict(export_params)
    export_params_local["scale"] = scale_m

    dataset_name = f"{dataset_prefix}_{day_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params_local)


def download_chirps_aggregate(tile, end_day, window_days, reducer, export_params, scale_m=5566, dataset_prefix="CHIRPS"):
    """Export a CHIRPS rolling-window aggregate ending at end_day for the given tile.

    Example dataset_name: CHIRPS_precip_14d_sum_2020_04_15

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        end_day: 'YYYY-MM-DD' or datetime/date
        window_days: e.g. 14
        reducer: 'sum' | 'mean' | 'max'
        export_params: dict used by ee.batch.Export.image.toDrive
        scale_m: export scale in meters (default ~5.5km)
    """
    end_str = _day_to_str(end_day)
    image = chirps_aggregate(end_day, window_days=window_days, reducer=reducer)

    export_params_local = dict(export_params)
    export_params_local["scale"] = scale_m

    dataset_name = f"{dataset_prefix}_precip_{int(window_days)}d_{reducer}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params_local)


# Example usage (uncomment):
# tile = kenya_tiles[0]
# download_chirps_for_day(tile, "2020-04-15", export_params)
# download_chirps_aggregate(tile, "2020-04-15", window_days=14, reducer="sum", export_params=export_params)


In [ ]:
def _normalize_date_list(dates):
    """Normalize a list of dates to unique 'YYYY-MM-DD' strings, sorted."""
    normalized = [_day_to_str(d) for d in dates]
    return sorted(set(normalized))


def schedule_chirps_exports_for_tile(
    tile,
    s1_dates,
    export_params,
    *,
    export_daily=True,
    aggregates=((1, "sum"), (3, "sum"), (7, "sum"), (14, "sum")),
    scale_m=5566,
    sleep_s=0,
):
    """Schedule CHIRPS exports for a single tile for a list of Sentinel-1 dates.

    Args:
        tile: (min_lon, min_lat, max_lon, max_lat)
        s1_dates: list of 'YYYY-MM-DD' strings or datetime/date
        export_params: base ee export params (region is set inside download_data_for_tile)
        export_daily: whether to export CHIRPS daily precip for each date
        aggregates: iterable of (window_days, reducer) to export, e.g. (14,'sum')
        scale_m: CHIRPS native export scale in meters (~5566)
        sleep_s: throttle between task submissions
    """
    dates = _normalize_date_list(s1_dates)

    for day_str in dates:
        if export_daily:
            download_chirps_for_day(tile, day_str, export_params, scale_m=scale_m, dataset_prefix="CHIRPS")
            if sleep_s:
                time.sleep(sleep_s)

        for window_days, reducer in aggregates:
            download_chirps_aggregate(
                tile,
                day_str,
                window_days=int(window_days),
                reducer=str(reducer),
                export_params=export_params,
                scale_m=scale_m,
                dataset_prefix="CHIRPS",
            )
            if sleep_s:
                time.sleep(sleep_s)


def schedule_chirps_exports_for_tiles(
    tiles,
    s1_dates,
    export_params,
    *,
    export_daily=True,
    aggregates=((1, "sum"), (3, "sum"), (7, "sum"), (14, "sum")),
    scale_m=5566,
    sleep_s=0,
):
    """Schedule CHIRPS exports for many tiles for a list of Sentinel-1 dates."""
    for tile in tiles:
        schedule_chirps_exports_for_tile(
            tile,
            s1_dates=s1_dates,
            export_params=export_params,
            export_daily=export_daily,
            aggregates=aggregates,
            scale_m=scale_m,
            sleep_s=sleep_s,
        )


# Example usage (uncomment):
# s1_dates = ["2020-04-15", "2020-04-21", "2020-05-03"]
# schedule_chirps_exports_for_tile(kenya_tiles[0], s1_dates, export_params, sleep_s=0)
# schedule_chirps_exports_for_tiles(kenya_tiles[:2], s1_dates, export_params, sleep_s=0)


## ERA5-Land soil moisture (hourly → daily snapshot + rolling aggregates)

Goal: for each Sentinel-1 acquisition date $t$, export soil moisture features aligned to the pass date.

- Default: **daily mean** of hourly ERA5-Land on date $t$ (robust when acquisition time is unknown)
- Optional: rolling-window means (e.g., 14-day mean ending at $t$) and lag-deltas (e.g., $SM(t)-SM(t-7)$)

In [ ]:
ERA5_LAND_HOURLY = "ECMWF/ERA5_LAND/HOURLY"

# Volumetric soil water (m3/m3) at 4 depth layers
ERA5_SM_BANDS = [
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "volumetric_soil_water_layer_4",
]


def era5_sm_daily_mean_image(day):
    """Daily mean soil moisture for the given UTC day.

    Uses hourly ERA5-Land, averages all hours within [day, day+1).

    Returns an ee.Image with bands: sm_l1, sm_l2, sm_l3, sm_l4
    """
    start = _to_ee_date(day)
    end = start.advance(1, "day")

    img = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end)
        .select(ERA5_SM_BANDS)
        .mean()
    )

    return ee.Image(img).rename(["sm_l1", "sm_l2", "sm_l3", "sm_l4"]).set({
        "date": start.format("YYYY-MM-dd"),
        "aggregation": "daily_mean",
    })


def era5_sm_aggregate(end_day, window_days, reducer="mean"):
    """Aggregate ERA5-Land hourly soil moisture over a rolling window ending at end_day (inclusive).

    Note: For soil moisture, 'mean' over the window is usually the most sensible.

    Args:
        end_day: window ends here (inclusive)
        window_days: number of days in window (>=1)
        reducer: 'mean' | 'max' | 'min'

    Returns:
        ee.Image with 4 bands named like: sm_l1_14d_mean, ...
    """
    if window_days < 1:
        raise ValueError("window_days must be >= 1")

    end = _to_ee_date(end_day)
    start = end.advance(-(window_days - 1), "day")

    ic = (
        ee.ImageCollection(ERA5_LAND_HOURLY)
        .filterDate(start, end.advance(1, "day"))
        .select(ERA5_SM_BANDS)
    )

    reducer = reducer.lower()
    if reducer == "mean":
        agg = ic.mean()
    elif reducer == "max":
        agg = ic.max()
    elif reducer == "min":
        agg = ic.min()
    else:
        raise ValueError("reducer must be one of: 'mean', 'max', 'min'")

    names = [f"sm_l{i}_{int(window_days)}d_{reducer}" for i in range(1, 5)]
    return ee.Image(agg).rename(names).set({
        "end_date": end.format("YYYY-MM-dd"),
        "start_date": start.format("YYYY-MM-dd"),
        "window_days": int(window_days),
        "reducer": reducer,
    })


def era5_sm_delta(end_day, lag_days):
    """Compute soil moisture change: daily_mean(end_day) - daily_mean(end_day - lag_days)."""
    if lag_days < 1:
        raise ValueError("lag_days must be >= 1")

    end = _to_ee_date(end_day)
    prev = end.advance(-lag_days, "day")

    img_end = era5_sm_daily_mean_image(end)
    img_prev = era5_sm_daily_mean_image(prev)

    delta = img_end.subtract(img_prev)
    names = [f"dsm_l{i}_{int(lag_days)}d" for i in range(1, 5)]
    return ee.Image(delta).rename(names).set({
        "end_date": end.format("YYYY-MM-dd"),
        "prev_date": prev.format("YYYY-MM-dd"),
        "lag_days": int(lag_days),
        "aggregation": "daily_mean_delta",
    })


def download_era5_sm_for_day(tile, day, export_params, scale_m=11132, dataset_prefix="ERA5_SM"):
    """Export daily-mean ERA5-Land soil moisture for a single day for the given 3°×3° tile.

    Notes:
      - We override scale to ERA5 native (~0.1° ≈ 11km). Do NOT export at 30m.
      - Uses existing download_data_for_tile().
    """
    day_str = _day_to_str(day)
    image = era5_sm_daily_mean_image(day)

    export_params_local = dict(export_params)
    export_params_local["scale"] = scale_m

    dataset_name = f"{dataset_prefix}_{day_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params_local)


def download_era5_sm_aggregate(tile, end_day, window_days, reducer, export_params, scale_m=11132, dataset_prefix="ERA5_SM"):
    """Export a rolling-window aggregate ending at end_day for the given tile."""
    end_str = _day_to_str(end_day)
    image = era5_sm_aggregate(end_day, window_days=window_days, reducer=reducer)

    export_params_local = dict(export_params)
    export_params_local["scale"] = scale_m

    dataset_name = f"{dataset_prefix}_{int(window_days)}d_{str(reducer)}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params_local)


def download_era5_sm_delta(tile, end_day, lag_days, export_params, scale_m=11132, dataset_prefix="ERA5_SM"):
    """Export daily-mean delta SM(t) - SM(t-lag) for the given tile."""
    end_str = _day_to_str(end_day)
    image = era5_sm_delta(end_day, lag_days=lag_days)

    export_params_local = dict(export_params)
    export_params_local["scale"] = scale_m

    dataset_name = f"{dataset_prefix}_d{int(lag_days)}_{end_str.replace('-', '_')}"
    return download_data_for_tile(tile, image, dataset_name, export_params_local)


def schedule_era5_sm_exports_for_tile(
    tile,
    s1_dates,
    export_params,
    *,
    export_daily=True,
    aggregates=((14, "mean"),),
    deltas=(7, 14),
    scale_m=11132,
    sleep_s=0,
):
    """Schedule ERA5 soil moisture exports for one tile for a list of S1 pass dates."""
    dates = _normalize_date_list(s1_dates)

    for day_str in dates:
        if export_daily:
            download_era5_sm_for_day(tile, day_str, export_params, scale_m=scale_m)
            if sleep_s:
                time.sleep(sleep_s)

        for window_days, reducer in aggregates:
            download_era5_sm_aggregate(
                tile,
                day_str,
                window_days=int(window_days),
                reducer=str(reducer),
                export_params=export_params,
                scale_m=scale_m,
            )
            if sleep_s:
                time.sleep(sleep_s)

        for lag_days in deltas:
            download_era5_sm_delta(
                tile,
                day_str,
                lag_days=int(lag_days),
                export_params=export_params,
                scale_m=scale_m,
            )
            if sleep_s:
                time.sleep(sleep_s)


def schedule_era5_sm_exports_for_tiles(
    tiles,
    s1_dates,
    export_params,
    *,
    export_daily=True,
    aggregates=((14, "mean"),),
    deltas=(7, 14),
    scale_m=11132,
    sleep_s=0,
):
    for tile in tiles:
        schedule_era5_sm_exports_for_tile(
            tile,
            s1_dates=s1_dates,
            export_params=export_params,
            export_daily=export_daily,
            aggregates=aggregates,
            deltas=deltas,
            scale_m=scale_m,
            sleep_s=sleep_s,
        )


# Example usage (uncomment):
# s1_dates = ["2020-04-15", "2020-04-21", "2020-05-03"]
# schedule_era5_sm_exports_for_tile(kenya_tiles[0], s1_dates, export_params, sleep_s=0)
# schedule_era5_sm_exports_for_tiles(kenya_tiles[:2], s1_dates, export_params, sleep_s=0)


In [43]:
check_task_status()

TASK DESCRIPTION               | STATE      | ID
------------------------------------------------------------
S06E033_CHIRPS_precip_14d_sum_2020_04_15 | READY      | 2NTCPEZLHKCJ5WMRUTMGZTDA
S06E033_CHIRPS_precip_2020_04_15 | RUNNING    | ISBU4IGWCFFL3CXJXS2URF5C


** read dates from parquet files!

In [62]:
# Check status of all export tasks
check_task_status(n=100)

TASK DESCRIPTION               | STATE      | ID
------------------------------------------------------------
S06E033_ERA5_SM_sm_14d_mean_2020_04_15 | RUNNING    | 6XCDJU2DZBGJKJ4IQ2OWJJZL
S06E033_ERA5_SM_daily_mean_2020_04_15 | RUNNING    | OJJTRCDNFJJ64AYNETFFHWJ3
S06E033_CHIRPS_precip_14d_sum_2020_04_15 | COMPLETED  | 2NTCPEZLHKCJ5WMRUTMGZTDA
S06E033_CHIRPS_precip_2020_04_15 | COMPLETED  | ISBU4IGWCFFL3CXJXS2URF5C


## For the tiles covering Kenya, export the dates where flood were detected.

In [2]:
from pathlib import Path
from huggingface_hub import hf_hub_download

def download_tile(tile,
                  repo_id = "ai-for-good-lab/ai4g-flood-dataset",
                  download_240m_buffer_tif = True,
                  download_80m_buffer_tif = True,
                  download_post_processing_parquet = True,
                  download_recurrence_80m_buffer_tif = True,
                  local_dir = None,
                  overwrite = True
                  ):
    
    print(f"\nDownloading data for tile: {tile}")
    
    if local_dir is None:
        local_dir = Path(f"flood_data")

    if not local_dir.exists():
        local_dir.mkdir(parents=True, exist_ok=True)

    root_dir = tile[:3]
    files_to_download = []
    if download_240m_buffer_tif:
        files_to_download.append(f"{root_dir}/{tile}/{tile}-240m-buffer.tif")

    if download_80m_buffer_tif:
        files_to_download.append(f"{root_dir}/{tile}/{tile}-80m-buffer.tif")

    if download_post_processing_parquet:
        files_to_download.append(f"{root_dir}/{tile}/{tile}-post-processing.parquet")

    if download_recurrence_80m_buffer_tif:
        files_to_download.append(f"{root_dir}/{tile}/{tile}-recurrence-80m-buffer.tif")

    for file_path in files_to_download:
        local_file_path = local_dir / Path(file_path)
        print(f"Does local file exist? {local_file_path.exists()}")
        if not local_file_path.exists() or overwrite:
            print(f"Downloading {file_path} to {local_file_path}")
            hf_hub_download(repo_id = repo_id,
                            filename = file_path,
                            local_dir = str(local_dir),
                            repo_type = "dataset",
            )
        else:
            print(f"File {local_file_path} already exists. Skipping download.")


    return


In [ ]:
# Generate list of tile names from kenya_tiles
kenya_bounds = get_country_bounds("Kenya")
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size = 3)
tile_names = [tile_name_from_latlon(tile[1], tile[0]) for tile in kenya_tiles]


for tile in tile_names:
    download_tile(
        tile = tile,
        download_240m_buffer_tif = True,
        download_80m_buffer_tif = False,
        download_post_processing_parquet = True,
        download_recurrence_80m_buffer_tif = False,
        local_dir = None,
        overwrite = False
    )
    time.sleep(3)  # Sleep for 3 seconds between downloads to avoid rate limiting
    


    


Does local file exist? True
File flood_data/S06/S06E033/S06E033-240m-buffer.tif already exists. Skipping download.
Does local file exist? True
File flood_data/S06/S06E033/S06E033-post-processing.parquet already exists. Skipping download.

Does local file exist? True
File flood_data/S06/S06E036/S06E036-240m-buffer.tif already exists. Skipping download.
Does local file exist? True
File flood_data/S06/S06E036/S06E036-post-processing.parquet already exists. Skipping download.

Does local file exist? True
File flood_data/S06/S06E039/S06E039-240m-buffer.tif already exists. Skipping download.
Does local file exist? True
File flood_data/S06/S06E039/S06E039-post-processing.parquet already exists. Skipping download.

Does local file exist? True
File flood_data/S03/S03E033/S03E033-240m-buffer.tif already exists. Skipping download.
Does local file exist? True
File flood_data/S03/S03E033/S03E033-post-processing.parquet already exists. Skipping download.

Does local file exist? True
File flood_data

In [49]:
import pandas as pd
from pathlib import Path

pp = Path("flood_data/S03/S03E033/S03E033-post-processing.parquet")

# Load the parquet file into a pandas DataFrame
df = pd.read_parquet(parquet_file)

# Find max month in the 'month' column
max_month = df['month'].max()
print(f"Max month in the data: {max_month}")

Max month in the data: 12


In [50]:
df.head()

,year,month,day,lat,lon,filename,land_cover,dem_metric_1,dem_metric_2,soil_moisture,soil_moisture_zscore,soil_moisture_sca,soil_moisture_zscore_sca,temp,edge_false_positives
0,2016,5,19,0.000016,33.662598,S1A_IW_GRDH_1SSV_20160519T032745_20160519T0328...,30,1.06,2.31,-99.000000,-99.00,39.000000,1.49,9999.0,0
1,2016,8,9,0.000034,33.678329,S1A_IW_GRDH_1SSV_20160809T160420_20160809T1604...,10,4.73,6.66,-99.000000,-99.00,31.700001,0.54,9999.0,1
2,2018,9,4,0.000041,34.145412,S1A_IW_GRDH_1SDV_20180904T160431_20180904T1604...,20,0.33,0.61,28.700001,0.56,44.599998,1.11,9999.0,1
3,2018,9,4,0.000041,34.147038,S1A_IW_GRDH_1SDV_20180904T160431_20180904T1604...,40,0.42,0.54,28.700001,0.56,44.599998,1.11,9999.0,1
4,2018,9,4,0.000041,34.147217,S1A_IW_GRDH_1SDV_20180904T160431_20180904T1604...,40,0.13,0.58,28.700001,0.56,44.599998,1.11,9999.0,1


In [4]:
# Define parquet filters
filter_params = {
    "dem_metric_2_max": 10,
    "soil_moisture_sca_min": 1,
    "soil_moisture_zscore_min": 1,
    "soil_moisture_min": 20,
    "temp_min": 0,
    "exclude_land_cover": 60,
    "edge_fp_eq": 0
}

In [ ]:
import pandas as pd
def extract_dates_from_parquet(parquet_file_path, filter_params = None):
    """Extract dates when floods were detected from a post-processing parquet file."""
    df = pd.read_parquet(parquet_file_path)

    # Create a 'date' column from 'year', 'month', 'day'
    df['date'] = pd.to_datetime(df[['year', 'month', 'day']])

    if filter_params is not None:
        print(f"\nApplying filters to data...")
        
        mask = (
            (df.dem_metric_2 < filter_params["dem_metric_2_max"]) &
            (df.soil_moisture_sca > filter_params["soil_moisture_sca_min"]) &
            (df.soil_moisture_zscore > filter_params["soil_moisture_zscore_min"]) &
            (df.soil_moisture > filter_params["soil_moisture_min"]) &
            (df.temp > filter_params["temp_min"]) &
            (df.land_cover != filter_params["exclude_land_cover"]) &
            (df.edge_false_positives == filter_params["edge_fp_eq"])
        )

        filtered_df = df[mask]

    else:
        filtered_df = df

    
    

    # Extract unique dates when floods were detected
    flood_dates = filtered_df['date'].unique()
    flood_dates = pd.to_datetime(flood_dates).strftime('%Y-%m-%d').tolist()
    print(f"Extracted {len(flood_dates)} unique flood dates from {parquet_file_path.name} after filtering.")

    # Save extracted dates to csv file
    output_csv_path = parquet_file_path.with_name(parquet_file_path.stem + '_flood_dates.csv')
    pd.DataFrame({'date': flood_dates}).to_csv(output_csv_path, index=False)
    print(f"Saved extracted flood dates to {output_csv_path}")

    del df, filtered_df, flood_dates

    return True



In [60]:
parquet_file.as_posix()

'flood_data/N00/N00E036/N00E036-post-processing.parquet'

In [53]:
df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
filtered_df = df[parquet_filters]
filtered_df.head()

,year,month,day,lat,lon,filename,land_cover,dem_metric_1,dem_metric_2,soil_moisture,soil_moisture_zscore,soil_moisture_sca,soil_moisture_zscore_sca,temp,edge_false_positives,date
19,2022,3,18,0.000072,35.222565,S1A_IW_GRDH_1SDV_20220318T155642_20220318T1557...,30,5.12,8.47,46.299999,1.23,6.900000,-0.62,9999.0,0,2022-03-18
41,2020,3,29,0.000090,34.161942,S1B_IW_GRDH_1SDV_20200329T032737_20200329T0328...,40,0.55,1.18,34.200001,1.19,34.900002,0.21,9999.0,0,2020-03-29
205,2016,4,1,0.000196,34.146721,S1A_IW_GRDH_1SSV_20160401T032740_20160401T0328...,20,0.47,0.54,39.500000,1.80,38.500000,0.55,9999.0,0,2016-04-01
256,2020,3,29,0.000270,34.161942,S1B_IW_GRDH_1SDV_20200329T032737_20200329T0328...,40,0.71,1.20,34.200001,1.19,34.900002,0.21,9999.0,0,2020-03-29
417,2016,4,1,0.000376,34.146721,S1A_IW_GRDH_1SSV_20160401T032740_20160401T0328...,30,0.12,0.54,39.500000,1.80,38.500000,0.55,9999.0,0,2016-04-01


In [21]:
filtered_df = df[parquet_filters]
filtered_df['date'] = pd.to_datetime(filtered_df[['year', 'month', 'day']])
filtered_df.head()


/tmp/ipykernel_3044927/375858114.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['date'] = pd.to_datetime(filtered_df[['year', 'month', 'day']])


,year,month,day,lat,lon,filename,land_cover,dem_metric_1,dem_metric_2,soil_moisture,soil_moisture_zscore,soil_moisture_sca,soil_moisture_zscore_sca,temp,edge_false_positives,date
2272,2015,4,17,-2.999947,35.300545,S1A_IW_GRDH_1SDV_20150417T160316_20150417T1603...,30,0.58,1.61,43.500000,5.56,10.6,1.98,15.000000,0,2015-04-17
2282,2020,2,2,-2.999941,34.146740,S1A_IW_GRDH_1SDV_20200202T160411_20200202T1604...,40,1.05,1.64,23.299999,1.23,18.4,1.38,17.799999,0,2020-02-02
2283,2020,2,2,-2.999941,34.146919,S1A_IW_GRDH_1SDV_20200202T160411_20200202T1604...,40,1.11,1.94,23.299999,1.23,18.4,1.38,17.799999,0,2020-02-02
2284,2020,2,2,-2.999941,34.147282,S1A_IW_GRDH_1SDV_20200202T160411_20200202T1604...,40,1.31,1.99,23.299999,1.23,18.4,1.38,17.799999,0,2020-02-02
2285,2020,2,2,-2.999941,34.147640,S1A_IW_GRDH_1SDV_20200202T160411_20200202T1604...,40,1.10,1.99,23.299999,1.23,18.4,1.38,17.799999,0,2020-02-02


In [54]:
flood_dates = filtered_df['date'].unique()
flood_dates = pd.to_datetime(flood_dates).strftime('%Y-%m-%d')
flood_dates, len(flood_dates)

(array(['2022-03-18', '2020-03-29', '2016-04-01', '2015-11-09',
        '2014-11-12', '2018-07-26', '2015-07-19', '2024-06-16',
        '2016-03-08', '2023-05-05', '2019-09-25', '2017-07-30',
        '2017-04-18', '2020-07-15', '2021-07-26', '2018-03-20',
        '2018-04-13', '2017-09-17', '2020-07-07', '2021-03-16',
        '2024-04-12', '2020-04-29', '2019-05-09', '2016-09-16',
        '2020-04-21', '2017-01-31', '2016-11-10', '2017-11-03',
        '2018-03-03', '2018-08-31', '2020-05-08', '2024-04-17',
        '2014-11-21', '2017-09-28', '2017-08-24', '2022-09-07',
        '2018-10-17', '2018-10-01', '2018-06-20', '2019-12-04',
        '2019-05-02', '2021-10-06', '2023-03-18', '2020-06-09',
        '2020-10-19', '2022-08-02', '2021-05-23', '2017-05-12',
        '2023-12-07', '2021-10-14', '2015-06-01', '2020-11-28',
        '2020-09-25', '2021-10-30', '2016-05-02', '2019-07-13',
        '2024-08-15', '2019-09-01', '2024-01-12', '2019-11-10',
        '2021-04-05', '2018-05-07', '201

In [35]:
flood_dates.shape

(677,)

In [ ]:
# Extract path to parquet file
base_path = Path("flood_data")
parquet_files = list(base_path.glob("**/*-post-processing.parquet"))
print(f"Found {len(parquet_files)} parquet files in {base_path}")

# Apply the date extraction function to each parquet file
for parquet_file in parquet_files:
    extract_dates_from_parquet(parquet_file, parquet_filters = filter_params)
    # break

Found 12 parquet files in flood_data

Applying filters to data...
Extracted 709 unique flood dates from N00E033-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E033/N00E033-post-processing_flood_dates.csv

Applying filters to data...
Extracted 530 unique flood dates from N00E036-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E036/N00E036-post-processing_flood_dates.csv

Applying filters to data...
Extracted 189 unique flood dates from N00E039-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E039/N00E039-post-processing_flood_dates.csv

Applying filters to data...
Extracted 470 unique flood dates from N03E033-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N03/N03E033/N03E033-post-processing_flood_dates.csv

Applying filters to data...
Extracted 577 unique flood dates from N03E036-post-processing.parquet after filtering.
Saved e

In [9]:
# Extract path to parquet file
base_path = Path("flood_data")
parquet_files = list(base_path.glob("**/*-post-processing.parquet"))
print(f"Found {len(parquet_files)} parquet files in {base_path}")

# Apply the date extraction function to each parquet file
for parquet_file in parquet_files:
    extract_dates_from_parquet(parquet_file, parquet_filters = filter_params)
    # break

Found 12 parquet files in flood_data

Applying filters to data...
Extracted 709 unique flood dates from N00E033-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E033/N00E033-post-processing_flood_dates.csv

Applying filters to data...
Extracted 530 unique flood dates from N00E036-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E036/N00E036-post-processing_flood_dates.csv

Applying filters to data...
Extracted 189 unique flood dates from N00E039-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N00/N00E039/N00E039-post-processing_flood_dates.csv

Applying filters to data...
Extracted 470 unique flood dates from N03E033-post-processing.parquet after filtering.
Saved extracted flood dates to flood_data/N03/N03E033/N03E033-post-processing_flood_dates.csv

Applying filters to data...
Extracted 577 unique flood dates from N03E036-post-processing.parquet after filtering.
Saved e

In [38]:
parquet_files

[PosixPath('flood_data/N00/N00E033/N00E033-post-processing.parquet'),
 PosixPath('flood_data/N00/N00E036/N00E036-post-processing.parquet'),
 PosixPath('flood_data/N00/N00E039/N00E039-post-processing.parquet'),
 PosixPath('flood_data/N03/N03E033/N03E033-post-processing.parquet'),
 PosixPath('flood_data/N03/N03E036/N03E036-post-processing.parquet'),
 PosixPath('flood_data/N03/N03E039/N03E039-post-processing.parquet'),
 PosixPath('flood_data/S03/S03E033/S03E033-post-processing.parquet'),
 PosixPath('flood_data/S03/S03E036/S03E036-post-processing.parquet'),
 PosixPath('flood_data/S03/S03E039/S03E039-post-processing.parquet'),
 PosixPath('flood_data/S06/S06E033/S06E033-post-processing.parquet'),
 PosixPath('flood_data/S06/S06E036/S06E036-post-processing.parquet'),
 PosixPath('flood_data/S06/S06E039/S06E039-post-processing.parquet')]

## SChedule GEE exports for all flood dates per tile

For each Kenya tile, load the extracted flood dates CSV and schedule exports of all dynamic conditioning variables (CHIRPS, ERA5 soil moisture, temperature, runoff) aligned to those dates. Static layers (DEM, slope, MERIT Hydro) only need one export per tile.

**Each dataset is exported at its native resolution, we'll deal with resampling later**

**Export resolution conventions:**
- SRTM/static layers: 30m (`crsTransform` for 1 arc-sec)
- MERIT Hydro: ~90m (3 arc-sec native)
- CHIRPS: ~5.5 km (0.05° native)
- ERA5-Land: ~11 km (0.1° native)
- MODIS NDVI: 250 m (native)
- OpenLandMap soil properties: 250 m (native)


In [ ]:
# ── Resolution-specific export kwargs ────────────────────────────────────────────
# Each dataset is exported at its native resolution to avoid inflating file sizes
# with meaningless upsampled pixels.



# SRTM DEM and Slope
# 1 arc-second (approximately 30m)
SRTM_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'crs_transform': [0.0002777777777777778, 0, -180.0001388888889,
                      0, -0.0002777777777777778, 60.00013888888889]

}

# CHIRPS Precipitation
# 0.05° ≈ 5.5 km
CHIRPS_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'crs_transform': [0.05, 0, -180, 0, -0.05, 50]
}

# ERA5-Land: soil moisture, temperature, runoff, 
# 0.1° ≈ 11 km
ERA5_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'crs_transform': [0.1, 0, -180.05, 0, -0.1, 90.05]
}

# MODIS NDVI
# ~250 m
MODIS_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'scale': 231.65635826395825
}

# OpenLandMap clay content
# 250 m
SOIL_CLAY_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'crs_transform': [0.002083333, 0, -180, 0, -0.002083333, 87.37]
}

# MERIT Hydro flow accumulation and HAND
# 90 m
MERIT_HYDRO_EXPORT_KWARGS = {
    'crs': 'EPSG:4326',
    'crs_transform': [0.0008333333333333334, 0, -180.00041666666667,
                      0, -0.0008333333333333334, 84.99958333333333]
}







In [17]:
kenya_tiles

[(33, -6, 36, -3),
 (36, -6, 39, -3),
 (39, -6, 42, -3),
 (33, -3, 36, 0),
 (36, -3, 39, 0),
 (39, -3, 42, 0),
 (33, 0, 36, 3),
 (36, 0, 39, 3),
 (39, 0, 42, 3),
 (33, 3, 36, 6),
 (36, 3, 39, 6),
 (39, 3, 42, 6)]

In [ ]:
def tile_name_from_latlon(lat, lon):

    """
    Example usage:
    Nairobi coordinates: 0.0236° S, 37.9062° E
    print(f"Nairobi tile: {tile_name_from_latlon(lat = 1.29, lon = 36.82)}")
    """

    # Data is in 3 x 3 degree tiles so
     
    lat_tile = (lat // 3) * 3
    lon_tile = (lon // 3) * 3
    lat_tile, lon_tile = int(lat_tile), int(lon_tile)

    lat_prefix = 'N' if lat_tile >= 0 else 'S'
    lon_prefix = 'E' if lon_tile >= 0 else 'W' 

    return f"{lat_prefix}{abs(lat_tile):02d}{lon_prefix}{abs(lon_tile):03d}"

In [ ]:
from natsort import natsorted
from pathlib import Path

def tile_name_to_coord_bounds(tile_name):
    """Convert a tile name like 'N00E036' back to (min_lon, min_lat, max_lon, max_lat).
    
    Inverse of tile_name_from_latlon(). Tile names encode the SW corner;
    each tile spans 3° × 3°.
    """
    lat_prefix = tile_name[0]   # 'N' or 'S'
    lat_val = int(tile_name[1:3])  # e.g. '06' → 6
    lon_prefix = tile_name[3] # 'E' or 'W'
    lon_val = int(tile_name[4:7])  # e.g. '033' → 33

    min_lat = lat_val if lat_prefix == 'N' else -lat_val
    min_lon = lon_val if lon_prefix == 'E' else -lon_val

    max_lat = min_lat + 3
    max_lon = min_lon + 3

    return (min_lon, min_lat, max_lon, max_lat)

def load_flood_dates_for_tile(tile_name, base_path = Path("flood_data")):
    """Load the CSV of extracted flood dates for a tile.
    
    Returns:
        List of date strings ['2020-04-15', ...] sorted chronologically,
        or empty list if CSV not found.
    """
    root_dir = tile_name[:3]
    csv_path = base_path / root_dir / tile_name / f"{tile_name}-post-processing_flood_dates.csv"

    if not csv_path.exists():
        print(f"⚠️  Flood dates CSV not found for {tile_name}: {csv_path}")
        return []
    
    df = pd.read_csv(csv_path)
    dates = df['date'].tolist()
    dates = natsorted(list(set(dates)))  # Ensure uniqueness and sort

    print(f"  Loaded {len(dates)} flood dates for {tile_name} (range: {dates[0]} → {dates[-1]})")
    return dates


   

In [35]:
a = natsorted(["2020-04-15", "2020-04-21", "2020-05-03", "2020-04-15", "2020-05-03"])
list(set(a))

['2020-04-15', '2020-04-21', '2020-05-03']

In [28]:
tile_name_to_bounds(tile_names[0])

(33, -6, 36, -3)

In [ ]:
def schedule_exports_for_tile(
    tile_name,
    flood_dates,
    export_params,
    # -- Static layers (exported once, not date-specific) --
    export_dem_slope = True,
    export_merit_hydro = True,
    export_soil_clay = True,

    # -- Dynamic layers (exported per flood date) --
    export_chirps_precipitation = True,
    chirps_windows = ((3, "sum"), (7, "sum"), (14, "sum")),  # (window_days, reducer)
    export_era5_sm = True,
    era5_sm_windows = ((7, "mean"), (14, "mean")),
    export_era5_temp = True,
    era5_temp_windows=((3, "mean"), (7, "mean"), (14, "mean")),
    export_era5_runoff=True,
    era5_runoff_windows=((3, "sum"), (7, "sum"), (14, "sum")),
    export_modis_ndvi = True,
    modis_ndvi_lookback = 32,

    # ── Control ──
    sleep_between_dates=3,
    sleep_between_tasks=3,
    max_dates=None,



):
    """Schedule all GEE export tasks for one Kenya tile.

    Static layers are exported once. Dynamic layers are exported for each
    flood date in the provided list.

    Args:
        tile_name: e.g. 'N00E036'
        flood_dates: list of 'YYYY-MM-DD' strings
        export_params: base export params dict (folder, maxPixels, fileFormat)
        max_dates: if set, only process the first N dates (useful for testing)
    """
    tile_bounds = tile_name_to_coord_bounds(tile_name)
    n_dates = len(flood_dates)

    if max_dates is not None:
        flood_dates = flood_dates[:max_dates]
        print(f"⚠️  Limiting to first {max_dates} dates for testing")

    print(f"\n{'='*70}")
    print(f"  Tile: {tile_name}  |  Bounds: {tile_bounds}  |  Flood dates: {len(flood_dates)}/{n_dates}")
    print(f"{'='*70}")

    # ── 1. Static layers (one export per tile) ───────────────────────────────
    if export_dem_slope:
        print(f"\n  [Static] DEM + Slope")
        download_elevation_and_slope(tile_bounds, export_params, kwargs = SRTM_EXPORT_KWARGS)
        if sleep_between_tasks:
            time.sleep(sleep_between_tasks)

    if export_merit_hydro:
        print(f"\n  [Static] MERIT Hydro (flow accumulation + HAND)")
        download_merit_hydro_layer(tile_bounds, export_params, layer_name = 'flow_accumulation', kwargs = MERIT_HYDRO_EXPORT_KWARGS)
        download_merit_hydro_layer(tile_bounds, export_params, layer_name = 'hand', kwargs = MERIT_HYDRO_EXPORT_KWARGS)
        if sleep_between_tasks:
            time.sleep(sleep_between_tasks)

    if export_soil_clay:
        print(f"\n  [Static] Soil clay content (OpenLandMap)")
        download_soil_static_property(tile_bounds, export_params, property_name = 'clay', kwargs = SOIL_CLAY_EXPORT_KWARGS)
        if sleep_between_tasks:
            time.sleep(sleep_between_tasks)

    # ── 2. Dynamic layers (per flood date) ───────────────────────────────────
    print(f"\n  [Dynamic] Scheduling exports for {len(flood_dates)} flood dates...")

    for i, day_str in enumerate(flood_dates):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"\n    Date {i+1}/{len(flood_dates)}: {day_str}")

        # ── CHIRPS precipitation ──
        if export_chirps_precipitation:
            # Daily precipitation
            download_chirps_precipitation_for_day(tile_bounds, day_str, export_params, kwargs = CHIRPS_EXPORT_KWARGS)

            # Rolling window aggregates
            for window_days, reducer in chirps_windows:
                download_chirps_precipitation_aggregate(tile_bounds, day_str, window_days=window_days, reducer=reducer,
                                                        export_params=export_params, kwargs = CHIRPS_EXPORT_KWARGS)

        # ── ERA5 soil moisture ──    
        if export_era5_sm:
            download_era5_sm_for_day(tile_bounds, day_str, export_params, kwargs = ERA5_EXPORT_KWARGS)
            for window_days, reducer in era5_sm_windows:
                download_era5_sm_aggregate(tile_bounds, day_str, window_days = window_days, reducer = reducer,
                                            export_params = export_params, kwargs = ERA5_EXPORT_KWARGS)
                
        

        # ── ERA5 temperature ──
        if export_era5_temp:
            download_era5_temperature_for_day(tile_bounds, day_str, export_params, kwargs = ERA5_EXPORT_KWARGS)
            for window_days, reducer in era5_temp_windows:
                download_era5_temperature_aggregate(tile_bounds, day_str, window_days = window_days, reducer = reducer,
                                            export_params = export_params, kwargs = ERA5_EXPORT_KWARGS)
                
        
        # ── ERA5 runoff ──
        if export_era5_runoff:
            download_era5_runoff_for_day(tile_bounds, day_str, export_params, kwargs = ERA5_EXPORT_KWARGS)
            for window_days, reducer in era5_runoff_windows:
                download_era5_runoff_aggregate(tile_bounds, day_str, window_days = window_days, reducer = reducer,
                                            export_params = export_params, kwargs = ERA5_EXPORT_KWARGS)
                
        # ── MODIS NDVI ──
        if export_modis_ndvi:
            download_modis_ndvi_for_day(tile_bounds, day_str, export_params, lookback_days = modis_ndvi_lookback, kwargs = MODIS_EXPORT_KWARGS)

        if sleep_between_dates:
            time.sleep(sleep_between_dates)


    print(f"\n  ✅ Done scheduling exports for {tile_name}")
            

        



    
        




    

    

In [ ]:
def schedule_all_kenya_exports(
    export_params,
    base_path = Path("flood_data"),
    tile_names = None,
    max_dates_per_tile = None,
    sleep_between_tiles = 5,
    **kwargs
):
    """Schedule GEE export tasks for all Kenya tiles.

    Args:
        export_params: base export params dict
        base_path: path to local flood_data directory with parquet/CSVs
        tile_names: list of tile name strings, or None to auto-detect from CSVs
        max_dates_per_tile: limit dates per tile (for testing)
        **kwargs: forwarded to schedule_exports_for_tile
    """

    if tile_names is None:
        # Auto-detect tiles from existing flood date CSVs
        csv_files = list(base_path.glob("**/*-post-processing_flood_dates.csv"))
        tile_names = natsorted([f.parent.name for f in csv_files])
        print(f"Auto-detected {len(tile_names)} tiles: {tile_names}")

    for tile_name in tile_names:
        flood_dates = load_flood_dates_for_tile(tile_name, base_path=base_path)
        if not flood_dates:
            print(f"⚠️  No flood dates found for {tile_name}. Skipping export scheduling.")
            continue

        schedule_exports_for_tile(
            tile_name = tile_name,
            flood_dates = flood_dates,
            export_params = export_params,
            max_dates = max_dates_per_tile,
            **kwargs
        )

        if sleep_between_tiles:
            time.sleep(sleep_between_tiles)


    print(f"\n{'='*70}")
    print(f"  All tiles scheduled. Use check_task_status() to monitor progress.")
    print(f"{'='*70}")

A Flux	A total amount that "fell" or "flowed" over time.	Sum	Runoff, Precipitation, Snowmelt.
A State	A "snapshot" of a condition (how hot, how wet, how fast).	Mean	Soil Water, Temperature, Wind Speed, Pressure.

In [30]:
# Sanity check: convert tile name to bounds and back
for tile in kenya_tiles:
    name = tile_name_from_latlon(tile[1], tile[0])
    bounds = tile_name_to_coord_bounds(name)
    print(f"Tile: {tile}, Name: {name}, Bounds from name: {bounds}")
    assert bounds == tile, f"Bounds mismatch: {bounds} vs {tile}"

Tile: (33, -6, 36, -3), Name: S06E033, Bounds from name: (33, -6, 36, -3)
Tile: (36, -6, 39, -3), Name: S06E036, Bounds from name: (36, -6, 39, -3)
Tile: (39, -6, 42, -3), Name: S06E039, Bounds from name: (39, -6, 42, -3)
Tile: (33, -3, 36, 0), Name: S03E033, Bounds from name: (33, -3, 36, 0)
Tile: (36, -3, 39, 0), Name: S03E036, Bounds from name: (36, -3, 39, 0)
Tile: (39, -3, 42, 0), Name: S03E039, Bounds from name: (39, -3, 42, 0)
Tile: (33, 0, 36, 3), Name: N00E033, Bounds from name: (33, 0, 36, 3)
Tile: (36, 0, 39, 3), Name: N00E036, Bounds from name: (36, 0, 39, 3)
Tile: (39, 0, 42, 3), Name: N00E039, Bounds from name: (39, 0, 42, 3)
Tile: (33, 3, 36, 6), Name: N03E033, Bounds from name: (33, 3, 36, 6)
Tile: (36, 3, 39, 6), Name: N03E036, Bounds from name: (36, 3, 39, 6)
Tile: (39, 3, 42, 6), Name: N03E039, Bounds from name: (39, 3, 42, 6)


In [ ]:
kenya_tiles, tile_names    

([(33, -6, 36, -3),
  (36, -6, 39, -3),
  (39, -6, 42, -3),
  (33, -3, 36, 0),
  (36, -3, 39, 0),
  (39, -3, 42, 0),
  (33, 0, 36, 3),
  (36, 0, 39, 3),
  (39, 0, 42, 3),
  (33, 3, 36, 6),
  (36, 3, 39, 6),
  (39, 3, 42, 6)],
 ['S06E033',
  'S06E036',
  'S06E039',
  'S03E033',
  'S03E036',
  'S03E039',
  'N00E033',
  'N00E036',
  'N00E039',
  'N03E033',
  'N03E036',
  'N03E039'])

In [20]:
# Generate list of tile names from kenya_tiles
kenya_bounds = get_country_bounds("Kenya")
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size = 3)
tile_names = [tile_name_from_latlon(tile[1], tile[0]) for tile in kenya_tiles]

In [14]:
0.0008333333333333334*111320

92.76666666666667

In [3]:
0.0002777777777777778* 111320, 0.05 * 111320, 0.1 * 111320

(30.92222222222222, 5566.0, 11132.0)

In [15]:
kenya_tiles

NameError: name 'kenya_tiles' is not defined

In [3]:
import ee
ee.Initialize(project = 'ee-erikkuwanjau')

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [ ]:
import ee

ee.Authenticate(auth_mode='notebook')